In [1]:
import os
import numpy as np
import cv2
import glob
from distutils.archive_util import make_archive
from imutils.object_detection import non_max_suppression
from multiprocessing import Pool
from tqdm.auto import tqdm

In [ ]:
"""
src: https://pyimagesearch.com/2018/08/20/opencv-text-detection-east-text-detector/
"""
#source_dir = '../data/hateful_memes/img'
#source_dir = '../data/hateful_memes/img'
#target_dir_masked = '../data/hateful_memes_masked'
target_dir_inpainted = '../data/hateful_memes_inpainted'
source_dir = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img'
target_dir_masked = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/masked'
target_dir_inpainted = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/inpainted'
east_path = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/MODEL/frozen_east_text_detection.pb'

In [3]:
model_width = 640  
model_height = 640
min_confidence = 0.4

In [4]:
print("[INFO] loading EAST text detector...")
net = cv2.dnn.readNet(east_path)
layerNames = [
    "feature_fusion/Conv_7/Sigmoid",
    "feature_fusion/concat_3"
]


[INFO] loading EAST text detector...


In [5]:
supported_formats = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']

image_files = []
for format in supported_formats:
    image_files.extend(glob.glob(os.path.join(source_dir, format)))

print(f"找到 {len(image_files)} 张图片需要处理")

找到 10000 张图片需要处理


## 图片直接掩码（输出mask文件夹）

In [6]:
def transform_image(img_fp):
    global model_width, model_height, min_confidence
    
    # load the input image and grab the image dimensions
    image = cv2.imread(img_fp)
    if image is None:
        print(f"无法读取图片: {img_fp}")
        return None
        
    masked = image.copy()
    (H, W) = image.shape[:2]
    
    # set the new width and height and then determine the ratio in change
    # for both the width and height
    (newW, newH) = (model_width, model_height)  # 修正变量名
    rW = W / float(newW)
    rH = H / float(newH)
    
    # resize the image and grab the new image dimensions
    image = cv2.resize(image, (newW, newH))
    (H, W) = image.shape[:2]

    # construct a blob from the image and then perform a forward pass of
    # the model to obtain the two output layer sets
    blob = cv2.dnn.blobFromImage(image, 1.0, (W, H),
        (123.68, 116.78, 103.94), swapRB=True, crop=False)
    net.setInput(blob)
    (scores, geometry) = net.forward(layerNames)

    # grab the number of rows and columns from the scores volume, then
    # initialize our set of bounding box rectangles and corresponding
    # confidence scores
    (numRows, numCols) = scores.shape[2:4]
    rects = []
    confidences = []
    
    # loop over the number of rows
    for y in range(0, numRows):
        # extract the scores (probabilities), followed by the geometrical
        # data used to derive potential bounding box coordinates that
        # surround text
        scoresData = scores[0, 0, y]
        xData0 = geometry[0, 0, y]
        xData1 = geometry[0, 1, y]
        xData2 = geometry[0, 2, y]
        xData3 = geometry[0, 3, y]
        anglesData = geometry[0, 4, y]

        # loop over the number of columns
        for x in range(0, numCols):
            # if our score does not have sufficient probability, ignore it
            if scoresData[x] < min_confidence:
                continue
            # compute the offset factor as our resulting feature maps will
            # be 4x smaller than the input image
            (offsetX, offsetY) = (x * 4.0, y * 4.0)
            # extract the rotation angle for the prediction and then
            # compute the sin and cosine
            angle = anglesData[x]
            cos = np.cos(angle)
            sin = np.sin(angle)
            # use the geometry volume to derive the width and height of
            # the bounding box
            h = xData0[x] + xData2[x]
            w = xData1[x] + xData3[x]
            # compute both the starting and ending (x, y)-coordinates for
            # the text prediction bounding box
            endX = int(offsetX + (cos * xData1[x]) + (sin * xData2[x]))
            endY = int(offsetY - (sin * xData1[x]) + (cos * xData2[x]))
            startX = int(endX - w)
            startY = int(endY - h)
            # add the bounding box coordinates and probability score to
            # our respective lists
            rects.append((startX, startY, endX, endY))
            confidences.append(scoresData[x])

    # 合并同一行的文本框
    if len(rects) > 0:
        rects = np.array(rects)
        
        # 计算每个框的中心y坐标和高度
        centers_y = (rects[:, 1] + rects[:, 3]) / 2
        heights = rects[:, 3] - rects[:, 1]
        avg_height = np.mean(heights)
        
        # 根据y坐标和高度将框分组到不同的行
        rows = []
        current_row = [rects[0]]
        for i in range(1, len(rects)):
            if abs(centers_y[i] - centers_y[i-1]) < avg_height / 2:
                current_row.append(rects[i])
            else:
                rows.append(current_row)
                current_row = [rects[i]]
        rows.append(current_row)
        
        merged_boxes = []
        for row in rows:
            row = np.array(row)
            min_x = np.min(row[:, 0])
            min_y = np.min(row[:, 1])
            max_x = np.max(row[:, 2])
            max_y = np.max(row[:, 3])
            merged_boxes.append([min_x, min_y, max_x, max_y])
        
        boxes = merged_boxes
    else:
        boxes = []

    # loop over the bounding boxes
    for (startX, startY, endX, endY) in boxes:
        # scale the bounding box coordinates based on the respective
        # ratios
        startX = int(startX * rW)
        startY = int(startY * rH)
        endX = int(endX * rW)
        endY = int(endY * rH)
        # draw the bounding box on the image
        cv2.rectangle(masked, (startX, startY), (endX, endY), (127, 127, 127), -1)

    return masked

In [8]:
from tqdm import tqdm

for i, img_path in enumerate(tqdm(image_files, desc="处理图片")):
    try:
        tqdm.write(f"处理第 {i+1}/{len(image_files)} 张图片: {os.path.basename(img_path)}")

        result_image = transform_image(img_path)
        
        if result_image is not None:
            filename = os.path.basename(img_path)
            name, ext = os.path.splitext(filename)
            output_path = os.path.join(target_dir_masked, f"{name}_masked{ext}")

            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            cv2.imwrite(output_path, result_image)
        
    except Exception as e:
        tqdm.write(f"处理图片 {img_path} 时出错: {str(e)}")
        continue

print("所有图片处理完成！")

处理图片:   0%|          | 0/10000 [00:00<?, ?it/s]    

处理第 1/10000 张图片: 01235.png


处理图片:   0%|          | 1/10000 [00:00<1:14:44,  2.23it/s]    

处理第 2/10000 张图片: 01236.png


处理图片:   0%|          | 2/10000 [00:00<1:16:26,  2.18it/s]    

...

处理第 104/10000 张图片: 01937.png


处理图片:   1%|          | 104/10000 [00:45<1:10:04,  2.35it/s]    

处理第 105/10000 张图片: 01943.png


处理图片:   1%|          | 104/10000 [00:45<1:12:06,  2.29it/s]


KeyboardInterrupt: 

def transform_and_save_image(img_fn):
    img_fp = os.path.join(source_dir, img_fn)
    img_fp_masked = os.path.join(target_dir_masked, img_fn)
    # img_fp_inpainted = os.path.join(target_dir_inpainted, img_fn)

    img_masked, img_inpainted = transform_image(img_fp)
    cv2.imwrite(img_fp_masked, img_masked)
    # cv2.imwrite(img_fp_inpainted, img_inpainted)

with Pool(64) as pool:
    #pool.map(transform_and_save_image, img_fns)
    for _ in tqdm(pool.imap_unordered(transform_and_save_image, img_fns), total=len(img_fns)):
        pass


## 若中途中断，采取以下代码继续进行mask

In [9]:
from tqdm import tqdm

# 找到起始文件的位置
start_index = 0
target_filename = "34671"

for i, img_path in enumerate(image_files):
    filename = os.path.basename(img_path)
    name, ext = os.path.splitext(filename)
    
    if name == target_filename or name.startswith(target_filename + "_"):
        start_index = i
        tqdm.write(f"找到起始文件 {target_filename}，从第 {i+1} 张图片开始处理")
        break

# 从起始位置开始处理
for i in tqdm(range(start_index, len(image_files)), desc="处理图片", initial=start_index, total=len(image_files)):
    img_path = image_files[i]
    
    try:
        tqdm.write(f"处理第 {i+1}/{len(image_files)} 张图片: {os.path.basename(img_path)}")

        result_image = transform_image(img_path)
        
        if result_image is not None:
            filename = os.path.basename(img_path)
            name, ext = os.path.splitext(filename)
            output_path = os.path.join(target_dir_masked, f"{name}_masked{ext}")

            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            cv2.imwrite(output_path, result_image)
        
    except Exception as e:
        tqdm.write(f"处理图片 {img_path} 时出错: {str(e)}")
        continue

print("所有图片处理完成！")

找到起始文件 34671，从第 3387 张图片开始处理


处理图片:  34%|███▍      | 3386/10000 [00:00<?, ?it/s]    

处理第 3387/10000 张图片: 34671.png


处理图片:  34%|███▍      | 3387/10000 [00:00<43:12,  2.55it/s]    

处理第 3388/10000 张图片: 34675.png


...

处理第 9999/10000 张图片: 98762.png


处理图片: 100%|█████████▉| 9999/10000 [1:49:02<00:00,  1.07it/s]    

处理第 10000/10000 张图片: 98764.png


处理图片: 100%|██████████| 10000/10000 [1:49:02<00:00,  1.01it/s]

所有图片处理完成！


## 纯二值黑白掩码（用于inpaint）

In [14]:
import os
import cv2
import numpy as np

def create_binary_mask(img_fp):
    global model_width, model_height, min_confidence
    
    # 加载图像
    image = cv2.imread(img_fp)
    if image is None:
        print(f"无法读取图片: {img_fp}")
        return None
    
    (H, W) = image.shape[:2]
    
    # 设置新尺寸和比例
    (newW, newH) = (model_width, model_height)
    rW = W / float(newW)
    rH = H / float(newH)
    
    # 调整图像尺寸
    resized_image = cv2.resize(image, (newW, newH))
    (H, W) = resized_image.shape[:2]

    # 模型推理
    blob = cv2.dnn.blobFromImage(resized_image, 1.0, (W, H),
        (123.68, 116.78, 103.94), swapRB=True, crop=False)
    net.setInput(blob)
    (scores, geometry) = net.forward(layerNames)

    # 检测文本框
    (numRows, numCols) = scores.shape[2:4]
    rects = []
    confidences = []
    
    for y in range(0, numRows):
        scoresData = scores[0, 0, y]
        xData0 = geometry[0, 0, y]
        xData1 = geometry[0, 1, y]
        xData2 = geometry[0, 2, y]
        xData3 = geometry[0, 3, y]
        anglesData = geometry[0, 4, y]

        for x in range(0, numCols):
            if scoresData[x] < min_confidence:
                continue
            (offsetX, offsetY) = (x * 4.0, y * 4.0)
            angle = anglesData[x]
            cos = np.cos(angle)
            sin = np.sin(angle)
            h = xData0[x] + xData2[x]
            w = xData1[x] + xData3[x]
            endX = int(offsetX + (cos * xData1[x]) + (sin * xData2[x]))
            endY = int(offsetY - (sin * xData1[x]) + (cos * xData2[x]))
            startX = int(endX - w)
            startY = int(endY - h)
            rects.append((startX, startY, endX, endY))
            confidences.append(scoresData[x])

    # 合并文本框
    boxes = []
    if len(rects) > 0:
        rects = np.array(rects)
        centers_y = (rects[:, 1] + rects[:, 3]) / 2
        heights = rects[:, 3] - rects[:, 1]
        avg_height = np.mean(heights)
        
        rows = []
        current_row = [rects[0]]
        for i in range(1, len(rects)):
            if abs(centers_y[i] - centers_y[i-1]) < avg_height / 2:
                current_row.append(rects[i])
            else:
                rows.append(current_row)
                current_row = [rects[i]]
        rows.append(current_row)
        
        merged_boxes = []
        for row in rows:
            row = np.array(row)
            min_x = np.min(row[:, 0])
            min_y = np.min(row[:, 1])
            max_x = np.max(row[:, 2])
            max_y = np.max(row[:, 3])
            merged_boxes.append([min_x, min_y, max_x, max_y])
        boxes = merged_boxes

    # 创建纯二值掩码（黑色背景，白色掩码区域）
    binary_mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
    
    # 在掩码上绘制白色矩形
    for (startX, startY, endX, endY) in boxes:
        startX = int(startX * rW)
        startY = int(startY * rH)
        endX = int(endX * rW)
        endY = int(endY * rH)
        cv2.rectangle(binary_mask, (startX, startY), (endX, endY), 255, -1)
    
    return binary_mask

# 定义路径
source_dir = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/img'
target_dir_mask = 'C:/Users/gidle/Desktop/Year3/NLP/Project/Hateful memes challenge/DATA/mask'

# 创建目标目录（如果不存在）
os.makedirs(target_dir_mask, exist_ok=True)

# 支持的图片格式
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')

# 获取源目录中的所有图片文件
image_files = [f for f in os.listdir(source_dir) if f.lower().endswith(valid_extensions)]

print(f"找到 {len(image_files)} 张图片需要处理")

# 处理每张图片
for image_file in image_files:
    source_path = os.path.join(source_dir, image_file)
    binary_mask = create_binary_mask(source_path)
    
    if binary_mask is not None:
        # 构建输出文件路径
        filename_without_ext = os.path.splitext(image_file)[0]
        output_path = os.path.join(target_dir_mask, f"{filename_without_ext}.png")
        
        # 保存掩码图片
        cv2.imwrite(output_path, binary_mask)
        print(f"已处理: {image_file} -> {filename_without_ext}.png")
    else:
        print(f"处理失败: {image_file}")

print("所有图片处理完成!")

找到 10000 张图片需要处理
已处理: 01235.png -> 01235.png
已处理: 01236.png -> 01236.png
已处理: 01243.png -> 01243.png
已处理: 01245.png -> 01245.png
已处理: 01247.png -> 01247.png
已处理: 01256.png -> 01256.png
已处理: 01258.png -> 01258.png
已处理: 01264.png -> 01264.png
已处理: 01268.png -> 01268.png
已处理: 01269.png -> 01269.png
已处理: 01274.png -> 01274.png
已处理: 01275.png -> 01275.png
已处理: 01276.png -> 01276.png
已处理: 01284.png -> 01284.png
已处理: 01293.png -> 01293.png
已处理: 01295.png -> 01295.png
已处理: 01324.png -> 01324.png
已处理: 01325.png -> 01325.png
已处理: 01327.png -> 01327.png
已处理: 01329.png -> 01329.png
已处理: 01348.png -> 01348.png
已处理: 01349.png -> 01349.png
已处理: 01359.png -> 01359.png
已处理: 01364.png -> 01364.png
已处理: 01379.png -> 01379.png
已处理: 01382.png -> 01382.png
已处理: 01389.png -> 01389.png
已处理: 01392.png -> 01392.png
已处理: 01395.png -> 01395.png
已处理: 01423.png -> 01423.png
已处理: 01436.png -> 01436.png
已处理: 01439.png -> 01439.png
已处理: 01452.png -> 01452.png
已处理: 01456.png -> 01456.png
已处理: 01459.png -> 01459.png
已处理